# centennial bluff mission a


## 1. Organize data
Create a folder under `semantic_SfM/data` and organize your data following the structures below. 

Agisoft:
```
semantic_SfM/data
    ├── centennial_bluff/mission_a
        ├── DJI_photos
        │       ├── DJI_0000.JPG
        │       ├── DJI_0001.JPG
        │       ├── ...
        │       └── DJI_0999.JPG
        ├── SfM_products
        │       ├── a.xml
        │       ├── a.jpg
        │       ├── a.mtl
        │       ├── a.obj
        │       ├── a.las       
        │       └── a_downsampled.las   
        ├── segmentations
        └── associations

```

In [1]:
import os

scene_dir = '../data/centennial_bluff/mission_a'
pointcloud_path = os.path.join(scene_dir, 'SfM_products', 'a_downsampled_2.las')
associations_folder_path = os.path.join(scene_dir, 'associations_2')
segmentations_folder_path = os.path.join(scene_dir, 'segmentations')
photos_folder_path = os.path.join(scene_dir, 'DJI_photos')
camera_path = os.path.join(scene_dir, 'SfM_products', 'a.xml')
mesh_path = os.path.join(scene_dir, 'SfM_products', 'a.obj')

## 2. Create 2D Segmentation using SAM

In [2]:
from ssfm.image_segmentation import ImageSegmentation
import os

In [3]:
sam_params = {}
sam_params['model_name'] = 'sam2'
sam_params['model_path'] = '../semantic_SfM/sam2/sam2.1_hiera_large.pt'
sam_params['device'] = 'cuda:1'
sam_params['points_per_side'] = 128
sam_params['points_per_batch'] = 128
sam_params['crop_n_layers'] = 3


image_path_list = [os.path.join(photos_folder_path, image) for image in os.listdir(photos_folder_path)]

# sort images based on the values of keyimages in file names
image_path_list = sorted(image_path_list, key=lambda x: int(x.split('/')[-1].split('.')[0].split('_')[-1]))

image_list = [image for image in os.listdir(photos_folder_path)]

# sort images based on the values of keyimages in file names
image_list = sorted(image_list, key=lambda x: int(x.split('/')[-1].split('.')[0].split('_')[-1]))

# print the length of the image list
print(len(image_list))

801


In [4]:
run_segmentation = False

if run_segmentation:
    image_segmentor = ImageSegmentation(sam_params)   
    image_segmentor.set_distortion_correction(camera_path)
    image_segmentor.batch_predict(image_path_list, segmentations_folder_path, save_overlap=True, skip_existing=False)

## 3. Create projection associations

In [5]:
from ssfm.probabilistic_projection import *
import time

In [6]:
pointcloud_projector = PointcloudProjection(depth_filtering_threshold=0.05, effective_depth = np.inf)

In [7]:
pointcloud_projector.read_camera_parameters(camera_path)
pointcloud_projector.read_mesh(mesh_path)
pointcloud_projector.read_pointcloud(pointcloud_path)

In [9]:
pointcloud_projector.parallel_batch_project_joblib(image_list, associations_folder_path, num_workers=16, save_depth=True)

Processing frames: 100%|██████████| 801/801 [32:00<00:00,  2.40s/it]


In [ ]:
# simple segmentation filter
from ssfm.simple_mask_filter import HollowMaskFilter

config = { 
    "segmentation_folder_path": "../data/centennial_bluff/mission_a/segmentations/",
    "output_folder_path": "../data/centennial_bluff/mission_a/segmentations_filtered/",
    "num_processes": 32,
    "min_area": 50,
    "min_fill_ratio": 0.3
}
mask_filter = HollowMaskFilter()

mask_filter(config)


In [7]:
segmentations_folder_path = "../data/centennial_bluff/mission_a/segmentations_filtered"

In [10]:
segmentations_folder_path = "../data/centennial_bluff/mission_a/segmentations_class_filter"

In [11]:
# build keyimage associations
from ssfm.keyimage_associations_builder import *

In [12]:
smc_solver = KeyimageAssociationsBuilder(image_list, associations_folder_path, segmentations_folder_path)

In [13]:
smc_solver.build_associations()
None

100%|██████████| 801/801 [02:15<00:00,  5.91it/s]


In [14]:
smc_solver.build_graph(20, device='cuda:2')
smc_solver.add_camera_to_graph([camera_path], camera_type="Agisoft")

Building edges on GPU with 20 chunks took 11.822312831878662 seconds.
../data/centennial_bluff/mission_a/associations_2/graph_with_cameras.graphml


In [14]:
smc_solver.find_min_cover()

| Metric                                                       | Count      | Percentage           |
----------------------------------------------------------------------------------------------------
| Number of points not covered by any image                    | 2586       | 0.16                 |
| Number of points covered by less than or equal to 1 image    | 3949       | 0.24                 |
| Number of points covered by less than or equal to 3 images   | 7636       | 0.47                 |
| Number of points covered by less than or equal to 5 images   | 13546      | 0.84                 |


## 4. Estimate memory usage

In [15]:
from ssfm.memory_calculator import memory_calculator

In [16]:
# pointcloud file
las_file = pointcloud_path
# image file sample; this needs to be an original image even if patch images are used
image_file = os.path.join(photos_folder_path, image_list[0])
# number of images
num_images = len(image_list)
# number of segmentation ids for each point in the point cloud
num_segmentation_ids = 5

memory_calculator(las_file, image_file, num_images, num_segmentation_ids)

+----------------------------------------+----------------------+
|              Memory Type               | Memory Required (GB) |
+----------------------------------------+----------------------+
|      Segmentation for each image       | 0.037181854248046875 |
| Pixel2point association for each image | 0.07436370849609375  |
| Point2pixel association for each image | 0.006016470491886139 |
|                                        |                      |
|      Segmentation for all images       |  29.782665252685547  |
| Pixel2point association for all images |  59.565330505371094  |
| Point2pixel association for all images |  4.819192864000797   |
|          pc_segmentation_ids           | 0.030082352459430695 |
|         pc_segmentation_probs          | 0.030082352459430695 |
|          keyimage_association          |  1.2047982160001993  |
|                 Total                  |   95.4321515429765   |
+----------------------------------------+----------------------+


## 5. Run object registration

In [1]:
from ssfm.segmo3d import *
from ssfm.post_processing import *
import time

In [ ]:
obr = SegMo3D(pointcloud_path, segmentations_folder_path, associations_folder_path, image_list=image_list, using_graph=True, radius=2, decaying=1, scene_name='mission_a_2')

# Run object registration
obr.segmo3d(iou_threshold=0.5, save_semantics=True, explicit_background=True)

In [ ]:
image_id = 800
semantics_folder_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}.npy'.format(image_id))
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}.las'.format(image_id))
add_semantics_to_pointcloud(pointcloud_path, semantics_folder_path, save_las_path, remove_small_N=30, nearest_interpolation=0)
#add_semantics_to_pointcloud(pointcloud_path, semantics_folder_path, save_las_path)

Before removing small semantics: 
maximum of semantics:  2454278
number of unique semantics:  19053
After removing small semantics: 
number of unique semantics:  19053


In [ ]:
semantic_pc_file_path = save_las_path
post_processing = PostProcessing(semantic_pc_file_path)
post_processing.shuffle_semantic_ids(exclude_largest_semantic=False)
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}_shuffled.las'.format(image_id))
post_processing.save_semantic_pointcloud(save_las_path)

Number of unique semantics:  549


In [3]:
semantic_pc_file_path = save_las_path
post_processing = PostProcessing(semantic_pc_file_path)
post_processing.sort_semantic_ids(exclude_largest_semantic=False)
save_las_path = os.path.join(associations_folder_path, 'semantics', 'semantics_{}_sorted.las'.format(image_id))
post_processing.save_semantic_pointcloud(save_las_path)

NameError: name 'save_las_path' is not defined